# Grid Creation & Feature Engineering

Creates a 100 × 100 m analysis grid over Tel Aviv and computes spatial features for each cell.

| Feature | Source | Method |
|---|---|---|
| `mean_NDVI` | Sentinel-2 raster | zonal stats |
| `mean_LST` | Landsat raster | zonal stats |
| `pct_green` | NDVI > 0.3 | derived |
| `pct_built` | Building footprints | overlay area |
| `mean_building_height` | Buildings layer | spatial join |
| `building_density` | Buildings layer | spatial join |
| `dist_major_road` | OSM roads | sjoin_nearest |
| `dist_to_water` | OSM water | sjoin_nearest |

**Output:** `outputs/tel_aviv_grid_features.gpkg`

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from rasterstats import zonal_stats
from shapely.geometry import box
from pathlib import Path

DATA   = Path('../Layers')
OUT    = Path('../outputs')
OUT.mkdir(exist_ok=True)

CELL_SIZE = 100   # metres, EPSG:2039
print('Imports OK')

## 1 — Load Layers

In [ ]:
border    = gpd.read_file(DATA / 'tel_aviv_border.geojson')
buildings = gpd.read_file(DATA / 'tel_aviv_buildings_only_ITM.geojson')   # EPSG:2039
roads     = gpd.read_file(DATA / 'osm/tel_aviv_major_roads.geojson')
water     = gpd.read_file(DATA / 'osm/tel_aviv_water.geojson')

# Reproject everything to EPSG:2039 for metric operations
border_itm = border.to_crs(2039)
roads_itm  = roads.to_crs(2039)
water_itm  = water.to_crs(2039)

print(f'Buildings : {len(buildings):,}  (CRS {buildings.crs.to_epsg()})')
print(f'Roads     : {len(roads_itm):,}')
print(f'Water     : {len(water_itm):,}')

## 2 — Create 100 × 100 m Grid

In [ ]:
minx, miny, maxx, maxy = border_itm.total_bounds

xs = np.arange(minx, maxx, CELL_SIZE)
ys = np.arange(miny, maxy, CELL_SIZE)

cells = [box(x, y, x + CELL_SIZE, y + CELL_SIZE)
         for x in xs for y in ys]

grid = gpd.GeoDataFrame({'geometry': cells}, crs=2039)
grid = gpd.clip(grid, border_itm).reset_index(drop=True)
grid['cell_id'] = grid.index

print(f'Grid cells: {len(grid):,}')
print(f'Cell area : {CELL_SIZE}m × {CELL_SIZE}m = {CELL_SIZE**2:,} m²')

## 3 — Raster Features (NDVI + LST)

In [ ]:
# zonal_stats requires geometries in the same CRS as the raster (EPSG:4326)
grid_wgs84 = grid.to_crs(4326)

print('Computing NDVI zonal stats...')
ndvi_stats = zonal_stats(
    grid_wgs84,
    str(DATA / 'tel_aviv_ndvi_2026-05-29.tif'),
    stats=['mean'],
    nodata=-9999
)
grid['mean_NDVI'] = [s['mean'] for s in ndvi_stats]

print('Computing LST zonal stats...')
lst_stats = zonal_stats(
    grid_wgs84,
    str(DATA / 'tel_aviv_lst_median.tif'),
    stats=['mean'],
    nodata=-9999
)
grid['mean_LST'] = [s['mean'] for s in lst_stats]

# pct_green: fraction of pixels in cell where NDVI > 0.3
print('Computing pct_green...')
green_stats = zonal_stats(
    grid_wgs84,
    str(DATA / 'tel_aviv_ndvi_2026-05-29.tif'),
    stats=['count'],
    nodata=-9999,
    add_stats={'green': lambda x: float(np.sum(x > 0.3)) / float(x.count()) if x.count() > 0 else 0}
)
grid['pct_green'] = [s['green'] for s in green_stats]

print(f"NDVI  — mean: {grid['mean_NDVI'].mean():.3f}")
print(f"LST   — mean: {grid['mean_LST'].mean():.1f} °C")
print(f"Green — mean: {grid['pct_green'].mean():.2%}")

## 4 — Building Features (pct_built, density, height)

In [ ]:
# Intersect building footprints with grid cells to get per-cell stats
# Both layers are in EPSG:2039
print('Intersecting buildings with grid... (may take ~1 min)')
bld_in_cells = gpd.overlay(
    buildings[['height', 'geometry']],
    grid[['cell_id', 'geometry']],
    how='intersection'
)
bld_in_cells['fragment_area'] = bld_in_cells.geometry.area

# pct_built: total building footprint area / cell area
built_area = bld_in_cells.groupby('cell_id')['fragment_area'].sum()
grid['pct_built'] = (grid['cell_id'].map(built_area).fillna(0) / CELL_SIZE ** 2).clip(0, 1)

# mean building height and count (weighted by fragment area for height)
bld_in_cells['weighted_height'] = bld_in_cells['height'] * bld_in_cells['fragment_area']
sum_wh   = bld_in_cells.groupby('cell_id')['weighted_height'].sum()
sum_area = bld_in_cells.groupby('cell_id')['fragment_area'].sum()
grid['mean_building_height'] = (grid['cell_id'].map(sum_wh) / grid['cell_id'].map(sum_area)).fillna(0)

# building density = unique buildings touching cell (before overlay explodes fragments)
density = bld_in_cells.groupby('cell_id').size()
grid['building_density'] = grid['cell_id'].map(density).fillna(0).astype(int)

print(f"pct_built            — mean: {grid['pct_built'].mean():.2%}")
print(f"mean_building_height — mean: {grid['mean_building_height'].mean():.1f} m")
print(f"building_density     — mean: {grid['building_density'].mean():.1f}")

## 5 — Distance Features (roads, water)

In [ ]:
# Use cell centroids for distance calculations
grid_centroids = grid.copy()
grid_centroids['geometry'] = grid.geometry.centroid

print('Computing dist_major_road...')
road_dist = gpd.sjoin_nearest(
    grid_centroids[['cell_id', 'geometry']],
    roads_itm[['geometry']],
    how='left',
    distance_col='dist_major_road'
).drop_duplicates('cell_id').set_index('cell_id')['dist_major_road']
grid['dist_major_road'] = grid['cell_id'].map(road_dist).fillna(road_dist.max())

print('Computing dist_to_water...')
water_dist = gpd.sjoin_nearest(
    grid_centroids[['cell_id', 'geometry']],
    water_itm[['geometry']],
    how='left',
    distance_col='dist_to_water'
).drop_duplicates('cell_id').set_index('cell_id')['dist_to_water']
grid['dist_to_water'] = grid['cell_id'].map(water_dist).fillna(water_dist.max())

print(f"dist_major_road — mean: {grid['dist_major_road'].mean():.0f} m")
print(f"dist_to_water   — mean: {grid['dist_to_water'].mean():.0f} m")

## 6 — Heat Priority Score

In [ ]:
def minmax_norm(s):
    return (s - s.min()) / (s.max() - s.min())

# Drop cells with missing raster data (sea / border edge)
grid = grid.dropna(subset=['mean_LST', 'mean_NDVI']).reset_index(drop=True)

# Normalise components
norm_lst     = minmax_norm(grid['mean_LST'])           # higher = hotter
norm_ndvi    = minmax_norm(grid['mean_NDVI'])          # higher = greener
built_score  = minmax_norm(grid['pct_built'])          # higher = more built
road_score   = 1 - minmax_norm(grid['dist_major_road'])  # closer road = higher exposure

grid['heat_priority'] = norm_lst * (1 - norm_ndvi) * built_score * road_score
grid['heat_priority'] = minmax_norm(grid['heat_priority'])   # rescale 0–1

hotspot_threshold = grid['heat_priority'].quantile(0.8)
grid['is_hotspot'] = (grid['heat_priority'] >= hotspot_threshold).astype(int)

print(f"Grid cells        : {len(grid):,}")
print(f"Hotspot threshold : {hotspot_threshold:.3f}  (top 20%)")
print(f"Hotspot cells     : {grid['is_hotspot'].sum():,}")

## 7 — Save

In [ ]:
out_path = OUT / 'tel_aviv_grid_features.gpkg'
grid.to_file(out_path, driver='GPKG')
print(f'Saved: {out_path}')
print(f'Columns: {list(grid.columns)}')
grid[['mean_LST','mean_NDVI','pct_built','mean_building_height',
      'building_density','dist_major_road','dist_to_water','heat_priority']].describe().round(2)

## 8 — Visualise

In [ ]:
features_to_plot = {
    'mean_LST'            : ('RdYlBu_r', 'LST (°C)'),
    'mean_NDVI'           : ('RdYlGn',   'NDVI'),
    'pct_built'           : ('YlOrRd',   'Built-up %'),
    'mean_building_height': ('YlOrRd',   'Mean Building Height (m)'),
    'dist_major_road'     : ('YlGnBu',   'Dist to Major Road (m)'),
    'dist_to_water'       : ('YlGnBu',   'Dist to Water (m)'),
    'heat_priority'       : ('RdYlBu_r', 'Heat Priority Score'),
}

grid_wgs = grid.to_crs(4326)
border_wgs = border.to_crs(4326)

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
axes = axes.flatten()

for i, (col, (cmap, label)) in enumerate(features_to_plot.items()):
    ax = axes[i]
    grid_wgs.plot(column=col, cmap=cmap, legend=True, ax=ax,
                  legend_kwds={'shrink': 0.6, 'label': label})
    border_wgs.boundary.plot(ax=ax, color='black', linewidth=0.8)
    ax.set_title(label, fontsize=11)
    ax.axis('off')

# Hotspot map in last panel
ax = axes[7]
grid_wgs[grid_wgs['is_hotspot'] == 0].plot(ax=ax, color='#f0f0f0')
grid_wgs[grid_wgs['is_hotspot'] == 1].plot(ax=ax, color='#d73027')
border_wgs.boundary.plot(ax=ax, color='black', linewidth=0.8)
ax.set_title('Hotspots (top 20%)', fontsize=11)
ax.axis('off')

plt.suptitle('Tel Aviv — Grid Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT / 'grid_features_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/grid_features_overview.png')